# MeowMemo H2 experiment: see or rerun the results

**H2:** cleaning up audio (trimming silence, evening out volume, reducing noise) helps the model guess the situation
(brushing / waiting for food / isolation) for cats it has never heard, including in noisy rooms.
**Result: not supported.** Details in `H2_EXPERIMENT.md`.

Pick one option:

| Option | Run cells | Time | What it does |
| --- | --- | --- | --- |
| **A. See the results** | 1, 2 | about 1 min | Shows the committed results. Nothing is installed or computed. |
| **B. Quick check** | 1 to 7 with `MODE = 'quick'` | about 1 min (38 s on a 12-core laptop), plus installs | Reruns clean audio for `baseline` and `trim` and compares them with the committed numbers. |
| **C. Full rerun** | 1 to 7 with `MODE = 'full'` | 48 min on a 12-core laptop; likely several hours on Colab's free CPU | Reruns everything: 6 clean-up variants, with and without noisy-copy training, on clean audio and 3 noise levels. |

Runs on Google Colab or any Jupyter with Python 3.12. For option C on Colab, keep the tab open: idle sessions disconnect.
A rerun can differ from the committed numbers in the third decimal place because of software or hardware differences;
the conclusions should not change.

**Data credits:** CatMeows (Ludovico et al., Zenodo, doi:10.5281/zenodo.4008297; scientific, non-commercial use) and
ESC-50 (K. J. Piczak, 2015; CC BY-NC 3.0). The audio is downloaded by scripts and never stored in the repository.

In [ ]:
# 1. Get the code: clones the h2-preprocessing branch on Colab, or reuses the checkout this notebook is in.
import json, os, subprocess, sys
from pathlib import Path

REPO = 'https://github.com/huangwanyi1024/MeowMemo.git'
BRANCH = 'h2-preprocessing'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # hide TensorFlow start-up messages


def sh(*cmd):
    """Run a command, show its output as it arrives, and stop if it fails."""
    cmd = [str(c) for c in cmd]
    print('$', ' '.join(cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='', flush=True)
    if proc.wait():
        raise RuntimeError(f'Command failed with exit code {proc.returncode}: {" ".join(cmd)}')


here = Path.cwd()
if (here / 'h2_experiment.py').exists():
    WORK = here
elif (here / 'work' / 'meowmemo-local-ai' / 'h2_experiment.py').exists():
    WORK = here / 'work' / 'meowmemo-local-ai'
else:
    if not Path('MeowMemo').exists():
        sh('git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, 'MeowMemo')
    WORK = Path('MeowMemo/work/meowmemo-local-ai').resolve()
os.chdir(WORK)
print('Working in', WORK, '| Python', sys.version.split()[0])

## Option A: the committed results

In [ ]:
# 2. Show the committed results (no installs, no computing).
def show(rows, columns):
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(rows, columns=columns).set_index(columns[0]))
    except ImportError:
        widths = [max(len(str(x)) for x in col) for col in zip(columns, *rows)]
        for r in [columns] + rows:
            print('  '.join(str(x).ljust(w) for x, w in zip(r, widths)))


def results_table(report):
    conds = list(report['results'])
    rows = []
    for arm in report['results'][conds[0]]:
        row = [arm]
        for c in conds:
            v = report['results'][c].get(arm)
            vb = v and v.get('vs_baseline')
            if v is None:
                row.append('')
            elif not vb:
                row.append(f"{v['macro_f1']:.3f}")
            else:
                lo, hi = vb['ci95']
                row.append(f"{vb['gain']:+.3f} [{lo:+.3f}, {hi:+.3f}]{'  PASS' if vb['supported'] else ''}")
        rows.append(row)
    rows.append(['cat-sound check pass rate (baseline)'] +
                [f"{report['results'][c]['baseline']['cat_gate_pass_rate']:.0%}" for c in conds])
    show(rows, ['arm (baseline: macro-F1; others: gain vs baseline, 95% range)'] + conds)
    gains = [v['vs_baseline'] for c in conds for v in report['results'][c].values() if 'vs_baseline' in v]
    print(f"{len(gains)} comparisons: {sum(g['supported'] for g in gains)} pass, "
          f"{sum(g['ci95'][1] < 0 for g in gains)} clearly worse, rest inconclusive.")


R = json.loads(Path('artifacts/h2-results.json').read_text())
SC = json.loads(Path('artifacts/h2-shortcut-check.json').read_text())
print(R['hypothesis'])
print('Pass rule:', R['pass_rule'])
results_table(R)
print(f"\nClip-length shortcut: length alone scores macro-F1 {SC['original_length']['macro_f1']:.3f} "
      f"(p = {SC['original_length']['chance']['p_value']}); after trimming {SC['trimmed_length']['macro_f1']:.3f} "
      f"(chance is about {SC['original_length']['chance']['chance_macro_f1_median']:.2f}).")

## Options B and C: rerun the experiment

Cell 3 installs the exact package versions used for the committed run (a few minutes). pip may warn about other
packages Colab ships with; that is fine, because the experiment runs as a separate process and no restart is needed.

In [ ]:
# 3. Install the package versions used for the committed run.
if sys.version_info[:2] != (3, 12):
    print(f'Note: the committed run used Python 3.12; this is {sys.version.split()[0]}. '
          'If the install fails, these exact versions are not available for this Python.')
PACKAGES = ['tensorflow==2.21.0', 'tf-keras==2.21.0', 'scikit-learn==1.9.1', 'scipy==1.18.1',
            'numpy==2.5.4', 'soundfile==0.14.0', 'noisereduce==3.0.3']
sh(sys.executable, '-m', 'pip', 'install', '--quiet', *PACKAGES)

In [ ]:
# 4. Download and check the data: CatMeows (440 clips, Zenodo) and the 20 ESC-50 noise clips (GitHub).
sh(sys.executable, 'download_catmeows.py')
sh(sys.executable, 'download_noise.py')               # must end with "OK: identical to the noise used ..."
sh(sys.executable, 'h2_data_checks.py', 'noise', 'data/noise')  # no animal sounds in the noise

In [ ]:
# 5. Run the automated checks (no TensorFlow needed).
sh(sys.executable, 'test_h2.py')
sh(sys.executable, 'test_h2_data_checks.py')

In [ ]:
# 6. Run the experiment. MODE = 'quick' (clean audio, baseline and trim) or 'full' (everything).
MODE = 'quick'

OUT = Path('artifacts/h2-rerun') / f'{MODE}.json'  # git-ignored; the committed results are never overwritten
OUT.parent.mkdir(parents=True, exist_ok=True)
args = ['--variants', 'baseline', 'trim'] if MODE == 'quick' else ['--noise-dir', 'data/noise']
sh(sys.executable, 'h2_experiment.py', 'run', *args, '--out', OUT)

In [ ]:
# 7. Compare the rerun with the committed results.
mine = json.loads(OUT.read_text())
rows, diffs = [], []
for cond, arms in mine['results'].items():
    for arm, v in arms.items():
        ref = R['results'].get(cond, {}).get(arm)
        if not ref:
            continue
        d = v['macro_f1'] - ref['macro_f1']
        diffs.append(abs(d))
        same = (v.get('vs_baseline') or {}).get('supported') == (ref.get('vs_baseline') or {}).get('supported')
        rows.append([f'{cond} | {arm}', f"{ref['macro_f1']:.3f}", f"{v['macro_f1']:.3f}", f'{d:+.3f}',
                     'yes' if same else 'NO'])
show(rows, ['condition | arm', 'committed macro-F1', 'rerun macro-F1', 'difference', 'same pass/fail'])
worst = max(diffs)
if worst <= 0.005:
    print(f'Matches the committed results (largest difference {worst:.3f}).')
elif worst <= 0.02:
    print(f'Close to the committed results (largest difference {worst:.3f}); small software or hardware differences.')
else:
    print(f'Differs from the committed results (largest difference {worst:.3f}). Check that cell 4 printed "OK" '
          'and that cell 3 installed the listed versions.')

## What the committed results say

- No clean-up step or noisy-copy training reliably improved the model: 1 of 44 comparisons passed (likely chance)
  and 15 were clearly worse. **H2 is not supported.**
- Clip length alone predicts the situation almost as well as the model, so trimming silence removes a shortcut.
- Background noise makes the cat-sound check fail more often (77% of clean clips pass, 58% in heavy noise).
- The limit is the data (440 clips, 21 cats, 2 breeds, recorded in controlled settings), not audio quality.